# Séance 7 — LLM & Datavisualisation
### Cours "Visualisation de Données"

---

> **🎯 Objectifs de la séance**
> - Comprendre comment les LLMs peuvent assister le travail d'analyse et de visualisation
> - Maîtriser le prompting structuré pour obtenir des sorties JSON exploitables
> - Générer du code Plotly via un LLM local (Ollama)
> - Identifier les limites et les hallucinations des LLMs sur des données numériques

> **⏱️ Durée totale estimée : 2h30**
> - Partie 1 (Cours) : ~45 min
> - Partie 2 (TP guidé + autonomie) : ~1h45

---

## Prérequis techniques

Avant de commencer, assurez-vous que les éléments suivants sont installés :

```bash
# Installation des librairies Python
pip install ollama pandas plotly

# Ollama doit être installé et lancé localement
# Téléchargement : https://ollama.com
# Lancement du service : ollama serve
# Téléchargement du modèle : ollama pull llama3.2
```

**Dataset utilisé :** `job_postings.csv` — 25 114 offres d'emploi avec 17 colonnes (titre, localisation, salaire, compétences, industrie, etc.)

---
# PARTIE 1 — COURS THÉORIQUE
---

## 1.1 — Architecture d'un LLM

### Qu'est-ce qu'un LLM ?

Un **Large Language Model (LLM)** est un réseau de neurones entraîné sur d'immenses corpus de texte pour prédire le prochain token (unité de texte) dans une séquence. Cette capacité de prédiction donne l'illusion de "compréhension" et permet de générer du texte cohérent.

### Concepts clés

#### Tokens
Un **token** est l'unité de base que le modèle traite. Il ne correspond pas exactement à un mot :
- `"visualisation"` → 1 token
- `"job_postings"` → 2–3 tokens
- `"4512"` → 1–4 tokens selon le modèle

En moyenne, 1 token ≈ 0,75 mot en anglais, et légèrement moins en français.

#### Context Window (fenêtre de contexte)
La **fenêtre de contexte** est la quantité maximale de tokens que le modèle peut traiter en une seule fois (entrée + sortie). Pour `llama3.2`, elle est de **128 000 tokens** (~96 000 mots). Dépasser cette limite tronque le contexte, ce qui peut produire des réponses incohérentes.

#### Température
La **température** contrôle le caractère aléatoire de la génération :
- `température = 0.0` → réponses déterministes, toujours les mêmes
- `température = 0.7` → bon équilibre créativité/cohérence (valeur par défaut)
- `température = 1.5` → très créatif, parfois incohérent

Pour des tâches structurées (JSON, code), on préfère une **température basse (0.0–0.3)**.

#### Modèles locaux vs cloud

| Critère | Local (Ollama) | Cloud (OpenAI, Anthropic) |
|---|---|---|
| **Coût** | Gratuit | Payant (par token) |
| **Confidentialité** | Données restent en local | Données envoyées au serveur |
| **Performance** | Limitée au matériel local | Très haute (GPU cloud) |
| **Débit** | Lent sur CPU | Rapide |
| **Modèles dispo** | llama3.2, mistral, gemma… | GPT-4o, Claude Opus… |

Dans ce cours, on utilise **Ollama** pour des raisons de confidentialité et de coût zéro.

### Flux de traitement d'une requête

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE LLM                         │
│                                                             │
│  ┌──────────┐    ┌───────────┐    ┌─────────┐    ┌───────┐ │
│  │  TEXTE   │───▶│TOKENIZER  │───▶│  LLM    │───▶│TOKENS │ │
│  │ d'entrée │    │(découpage)│    │(réseau  │    │sortie │ │
│  │          │    │           │    │de neur.)│    │       │ │
│  └──────────┘    └───────────┘    └─────────┘    └───┬───┘ │
│                                                       │     │
│                                                       ▼     │
│                                               ┌────────────┐│
│                                               │DÉTOKENIZER ││
│                                               │(reconstr.) ││
│                                               └─────┬──────┘│
│                                                     │       │
│                                                     ▼       │
│                                               ┌──────────┐  │
│                                               │  TEXTE   │  │
│                                               │ de sortie│  │
│                                               └──────────┘  │
└─────────────────────────────────────────────────────────────┘
```

### Pourquoi la précision numérique est un risque majeur

Les LLMs **ne calculent pas** : ils **prédisent** du texte. Lorsqu'on demande `"Combien de lignes contient ce CSV ?"`, le modèle ne compte pas — il génère la réponse la plus probable statistiquement d'après son entraînement. Résultat :

- Il peut **inventer un nombre plausible** (hallucination numérique)
- Il peut **arrondir** de façon erronée
- Il peut **extrapoler** à partir d'exemples vus à l'entraînement

> **⚠️ Règle d'or : ne jamais faire confiance à un chiffre produit par un LLM sans vérification indépendante via pandas/numpy.**

## 1.2 — Prompting structuré pour la data

### Stratégies de prompting

#### Zero-shot
On donne l'instruction directement, sans exemple. Simple mais moins précis pour des tâches complexes.

```
Recommande une visualisation pour ce dataset.
```

#### Few-shot
On donne 1 à 5 exemples du format attendu avant la vraie question. Le modèle comprend mieux le format souhaité.

```
Exemple 1 : Dataset de ventes → Bar chart par région
Exemple 2 : Dataset de températures → Line chart temporel
Maintenant, pour ce dataset d'offres d'emploi :
```

#### Chain-of-thought (CoT)
On demande au modèle de **raisonner étape par étape** avant de répondre. Améliore significativement la qualité sur des tâches analytiques.

```
Réfléchis étape par étape :
1. Quelles sont les variables disponibles ?
2. Quelles questions analytiques sont pertinentes ?
3. Quelle visualisation répond le mieux à chaque question ?
```

### Forçage de sortie JSON structurée

Pour automatiser le traitement des réponses, on peut **forcer** le LLM à produire du JSON en :
1. Définissant le schéma attendu dans le prompt
2. Donnant un exemple de sortie
3. Terminant le prompt par `"Réponds UNIQUEMENT avec du JSON valide, sans explication."`

Exemple de prompt structuré :
```
Tu es un expert en datavisualisation. Analyse ce dataset et réponds
UNIQUEMENT avec un JSON valide ayant cette structure exacte :
{
  "recommandations": [
    {
      "type": "bar chart",
      "colonnes": ["col1", "col2"],
      "justification": "...",
      "priorite": 1
    }
  ]
}
```

### Exemple complet — Prompt structuré avec Ollama

La cellule suivante est une **démonstration complète** : chargement des données, envoi au LLM, parsing de la réponse JSON.

In [ ]:
# NE PAS MODIFIER - Démonstration
# Exemple complet : prompt structuré pour obtenir une réponse JSON

import ollama
import pandas as pd
import json
import re

# --- Chargement d'un petit échantillon du dataset ---
df = pd.read_csv('job_postings.csv')
sample = df.head(10)

# Résumé des colonnes pour le prompt (on n'envoie pas toutes les données brutes)
colonnes_info = ", ".join(df.columns.tolist())
apercu = sample.to_string(index=False, max_cols=8)

# --- Construction du prompt structuré ---
prompt = f"""Tu es un expert en datavisualisation. Voici un aperçu d'un dataset d'offres d'emploi.

Colonnes disponibles : {colonnes_info}

Aperçu des 10 premières lignes :
{apercu}

Analyse ce dataset et réponds UNIQUEMENT avec un JSON valide, sans aucune explication ni texte avant ou après.
Le JSON doit avoir cette structure exacte :
{{
  "analyse": {{
    "type_dataset": "description en une phrase",
    "variable_cle": "la colonne la plus intéressante pour analyser",
    "recommandation_principale": "type de visualisation recommandé",
    "justification": "explication en 2-3 phrases"
  }}
}}"""

# --- Appel à l'API Ollama ---
print("Envoi de la requête à Ollama (llama3.2)...")
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt}]
)

# --- Récupération du contenu texte ---
contenu_brut = response['message']['content']
print("Réponse brute du LLM :")
print(contenu_brut)
print("-" * 50)

# --- Parsing JSON robuste (le LLM peut envelopper le JSON dans du texte ou du markdown) ---
def extraire_json(texte):
    """Extrait et parse un bloc JSON depuis une réponse LLM potentiellement bruitée."""
    # Étape 1 : supprimer les balises markdown ```json ... ```
    texte_clean = re.sub(r"```(?:json)?\s*", "", texte).replace("```", "").strip()

    # Étape 2 : essai de parsing direct
    try:
        return json.loads(texte_clean)
    except json.JSONDecodeError:
        pass

    # Étape 3 : extraire le premier bloc {...} complet (gestion des accolades imbriquées)
    profondeur, debut = 0, -1
    for idx, car in enumerate(texte_clean):
        if car == "{":
            if profondeur == 0:
                debut = idx
            profondeur += 1
        elif car == "}":
            profondeur -= 1
            if profondeur == 0 and debut != -1:
                candidat = texte_clean[debut:idx + 1]
                try:
                    return json.loads(candidat)
                except json.JSONDecodeError:
                    pass  # bloc trouvé mais encore invalide
                break

    return None  # échec total

resultat = extraire_json(contenu_brut)
if resultat is None:
    print("⚠️  Impossible de parser le JSON — réponse brute affichée ci-dessus.")

# --- Affichage structuré ---
if resultat:
    analyse = resultat.get('analyse', {})
    print(f"Type de dataset    : {analyse.get('type_dataset', 'N/A')}")
    print(f"Variable clé       : {analyse.get('variable_cle', 'N/A')}")
    print(f"Recommandation     : {analyse.get('recommandation_principale', 'N/A')}")
    print(f"Justification      : {analyse.get('justification', 'N/A')}")

## 1.3 — Limites et biais des LLMs en analyse de données

### Hallucination numérique

Le phénomène d'**hallucination** désigne le fait qu'un LLM génère des informations fausses mais présentées avec confiance. Pour les données numériques, c'est particulièrement dangereux :

- **Comptage inventé** : `"Il y a environ 30 000 lignes"` alors qu'il y en a 25 114
- **Statistiques fabriquées** : `"Le salaire moyen est de 65 000$"` sans avoir calculé quoi que ce soit
- **Tendances imaginées** : `"Le secteur tech représente 45% des offres"` sans agrégation réelle

### Biais de sélection

Quand on envoie seulement un échantillon (`head(10)`) au LLM, il généralise à partir de cet échantillon :
- Si les 10 premières lignes sont toutes du secteur tech, le LLM pourrait conclure que c'est un dataset tech
- Les valeurs manquantes peuvent être interprétées comme une absence de données dans toute la colonne

### Limites de la fenêtre de contexte

Un CSV de 25 114 lignes fait plusieurs **mégaoctets** de texte, bien au-delà de ce qu'on peut envoyer efficacement :
- Envoyer 1 000 lignes ≈ 50 000–80 000 tokens (selon les données)
- Le coût de traitement augmente quadratiquement avec la taille du contexte
- La qualité de la réponse **décroît** si le contexte est trop long

**Bonne pratique :** Envoyer un résumé statistique (`df.describe()`, liste des colonnes, quelques exemples) plutôt que les données brutes.

### Démonstration : l'hallucination numérique en action

La cellule suivante illustre ce qui se passe quand on demande au LLM de "compter" quelque chose sans lui donner la vraie valeur.

In [ ]:
# NE PAS MODIFIER - Démonstration
# Illustration de l'hallucination numérique

import ollama
import pandas as pd

# Chargement du dataset pour connaître les vraies valeurs
df = pd.read_csv('job_postings.csv')

# --- Question au LLM SANS lui donner le nombre réel ---
prompt_sans_info = """J'ai un fichier CSV appelé job_postings.csv qui contient des offres d'emploi.
Sans que je te donne plus d'information, combien de lignes penses-tu qu'il contient ?
Donne un chiffre précis avec ta meilleure estimation."""

print("=" * 60)
print("QUESTION sans information réelle :")
print("=" * 60)
response_sans_info = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt_sans_info}]
)
print(response_sans_info['message']['content'])

print()
print("=" * 60)
print(f"VÉRITÉ TERRAIN (pandas) : {len(df)} lignes")
print("=" * 60)
print()

# --- Autre exemple : demander une moyenne sans données ---
prompt_moyenne = """Pour un dataset d'offres d'emploi avec des colonnes Minimum Pay et Maximum Pay,
quel est le salaire moyen approximatif en dollars américains ?
Donne un chiffre précis."""

print("DEMANDE de moyenne sans données réelles :")
response_moyenne = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt_moyenne}]
)
print(response_moyenne['message']['content'])

print()
# Calcul réel
salaire_moyen_min = df['Minimum Pay'].dropna().mean()
salaire_moyen_max = df['Maximum Pay'].dropna().mean()
print(f"VÉRITÉ TERRAIN (pandas) :")
print(f"  Salaire minimum moyen : {salaire_moyen_min:,.2f} $")
print(f"  Salaire maximum moyen : {salaire_moyen_max:,.2f} $")
print()
print(">>> CONCLUSION : Le LLM a inventé des chiffres. Toujours vérifier avec pandas !")

---
# PARTIE 2 — TRAVAUX PRATIQUES
---

> **💡 Conseil général**
> - Les sections **DÉMONSTRATION** : exécutez-les et lisez attentivement le code et les commentaires.
> - Les sections **À VOUS** : complétez les `# TODO` en vous inspirant des démonstrations.
> - Si le LLM est lent, c'est normal sur CPU — préparez votre réponse pendant qu'il génère.

---
## Exercice 1 — Setup et premier contact

> **🎯 Objectif :** Vérifier l'installation, charger le dataset, envoyer un premier prompt simple.

> **⏱️ Durée estimée : 20 minutes**

---
### 1-A — DÉMONSTRATION

In [ ]:
# NE PAS MODIFIER - Démonstration
# Étape 1 : Vérifier que Ollama est bien installé et fonctionnel

import ollama

try:
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': 'Réponds juste "Ollama fonctionne !" en français.'}]
    )
    print("Statut Ollama :", response['message']['content'])
except Exception as e:
    print(f"Erreur de connexion à Ollama : {e}")
    print("Vérifiez que le service Ollama est lancé ('ollama serve' dans un terminal).")

In [ ]:
# NE PAS MODIFIER - Démonstration
# Étape 2 : Charger le dataset et afficher ses informations de base

import pandas as pd

# Chargement du CSV
df = pd.read_csv('job_postings.csv')

print(f"Dataset chargé avec succès !")
print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print()
print("Colonnes disponibles :")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")
print()
print("Aperçu des 3 premières lignes :")
df.head(3)

In [ ]:
# NE PAS MODIFIER - Démonstration
# Étape 3 : Envoyer un prompt de description générale du dataset

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

# On construit un résumé compact à envoyer au LLM
# (on n'envoie PAS toutes les données pour ne pas dépasser le contexte)
resume_dataset = f"""Dataset : job_postings.csv
Nombre de lignes : {len(df)}
Colonnes : {', '.join(df.columns.tolist())}

Exemple de 3 lignes (colonnes sélectionnées) :
{df[['Job Title', 'Company Industry', 'Job Location', 'Minimum Pay', 'Maximum Pay', 'Job Skills']].head(3).to_string(index=False)}"""

prompt_description = f"""Voici les informations sur un dataset d'offres d'emploi :

{resume_dataset}

Décris brièvement (en 4-5 phrases en français) :
1. Ce que contient ce dataset
2. Les analyses les plus utiles qu'on pourrait faire avec
3. Les visualisations les plus pertinentes à créer"""

print("Envoi du prompt au LLM...")
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt_description}]
)
print("Réponse du LLM :")
print("-" * 60)
print(response['message']['content'])

---
### 1-B — À VOUS

> **📋 Consignes :**
> 1. Rédigez un prompt qui demande à Ollama quelle serait la **meilleure visualisation** pour ce dataset d'offres d'emploi
> 2. Décrivez les colonnes dans le prompt pour que le LLM ait du contexte
> 3. Affichez et analysez la réponse

> **💡 Conseil :** Incluez dans votre prompt les noms des colonnes et leurs types (catégoriel, numérique, textuel, date). Plus vous donnez de contexte, meilleure sera la réponse.

In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

# TODO : Construire une description des colonnes du dataset
# Indice : Décrivez chaque colonne avec son type de données et quelques valeurs d'exemple
# Exemple : "Job Title (texte) : ex. 'Data Analyst', 'Software Engineer'"
description_colonnes = """
# TODO : Complétez cette description colonne par colonne
# Colonnes disponibles dans df.columns :
# Job Posting ID, Job Posting Date, Job Title, Job Title Full, 
# Job Title Additional Info, Job Position Type, Job Position Level,
# Years of Experience, Job Skills, Job Location,
# Minimum Pay, Maximum Pay, Pay Rate, Number of Applicants,
# Company Name, Company Industry, Company Size
"""

# TODO : Construire le prompt
# Il doit :
# - Mentionner qu'il s'agit d'un dataset d'offres d'emploi
# - Inclure la description des colonnes
# - Demander explicitement quelle est la MEILLEURE visualisation à créer
# - Demander une justification
mon_prompt = """
# TODO : Écrivez votre prompt ici
"""

# TODO : Envoyer le prompt à Ollama
# Indice : utilisez ollama.chat() avec model='llama3.2'
# response = ...

# TODO : Extraire et afficher le contenu de la réponse
# Indice : response['message']['content']
# print(...)

---
## Exercice 2 — Recommandations de visualisations en JSON

> **🎯 Objectif :** Forcer le LLM à produire du JSON structuré et exploiter la réponse programmatiquement.

> **⏱️ Durée estimée : 40 minutes**

---
### 2-A — DÉMONSTRATION

In [ ]:
# NE PAS MODIFIER - Démonstration
# Obtenir 3 recommandations de visualisations au format JSON et les parser

import ollama
import pandas as pd
import json
import re

df = pd.read_csv('job_postings.csv')

# On prend un échantillon de 20 lignes pour le contexte
sample = df.head(20)
colonnes = df.columns.tolist()
apercu = sample[['Job Title', 'Company Industry', 'Job Location', 'Job Skills',
                  'Minimum Pay', 'Maximum Pay', 'Job Position Level']].head(5).to_string(index=False)

# Prompt avec forçage JSON
prompt_json = f"""Tu es un expert en datavisualisation avec Python et Plotly.
Voici un dataset d'offres d'emploi :

Colonnes : {', '.join(colonnes)}
Nombre de lignes : {len(df)}
Aperçu :
{apercu}

Propose exactement 3 recommandations de visualisations pertinentes pour analyser ce dataset.

Réponds UNIQUEMENT avec un JSON valide, sans aucun texte avant ou après le JSON.
Structure exacte à respecter :
{{
  "recommandations": [
    {{
      "numero": 1,
      "titre": "Titre descriptif de la visualisation",
      "type_chart": "bar chart | line chart | scatter plot | pie chart | heatmap | ...",
      "colonnes_x": "nom de la colonne pour l'axe X",
      "colonnes_y": "nom de la colonne pour l'axe Y",
      "couleur": "colonne optionnelle pour la couleur (ou null)",
      "question_analytique": "Quelle question cette viz répond-elle ?",
      "justification": "Pourquoi cette visualisation est-elle pertinente ?",
      "priorite": "haute | moyenne | basse"
    }}
  ]
}}"""

print("Envoi de la requête à Ollama...")
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt_json}]
)

contenu = response['message']['content']

# --- Parsing JSON robuste ---
resultat = None
try:
    # Tentative directe
    resultat = json.loads(contenu)
except json.JSONDecodeError:
    # Extraction du bloc JSON si le LLM a ajouté du texte
    match = re.search(r'\{.*\}', contenu, re.DOTALL)
    if match:
        try:
            resultat = json.loads(match.group())
        except json.JSONDecodeError:
            print("Impossible de parser le JSON. Réponse brute :")
            print(contenu)

# --- Affichage structuré des recommandations ---
if resultat:
    recommandations = resultat.get('recommandations', [])
    print(f"\n{len(recommandations)} recommandations reçues :")
    print("=" * 60)
    for reco in recommandations:
        print(f"\n[{reco.get('numero', '?')}] {reco.get('titre', 'Sans titre')}")
        print(f"    Type       : {reco.get('type_chart', 'N/A')}")
        print(f"    Axe X      : {reco.get('colonnes_x', 'N/A')}")
        print(f"    Axe Y      : {reco.get('colonnes_y', 'N/A')}")
        print(f"    Couleur    : {reco.get('couleur', 'N/A')}")
        print(f"    Question   : {reco.get('question_analytique', 'N/A')}")
        print(f"    Priorité   : {reco.get('priorite', 'N/A')}")
        print(f"    Justif.    : {reco.get('justification', 'N/A')}")

---
### 2-B — À VOUS

> **📋 Consignes :**
> 1. Modifiez le prompt pour demander **5 recommandations** au lieu de 3
> 2. Ajoutez une contrainte dans le prompt : *"Recommande uniquement des graphiques montrant une évolution temporelle ou une comparaison entre industries"*
> 3. Parsez et affichez les résultats de façon claire

> **💡 Conseil :** Copiez la structure du prompt de démonstration et modifiez uniquement les paramètres demandés. La structure JSON peut rester identique.

In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd
import json
import re

df = pd.read_csv('job_postings.csv')
colonnes = df.columns.tolist()
apercu = df[['Job Title', 'Company Industry', 'Job Location', 'Minimum Pay', 'Maximum Pay']].head(5).to_string(index=False)

# TODO : Modifier le prompt pour :
#   1. Demander 5 recommandations (pas 3)
#   2. Ajouter la contrainte de type de visualisation (temporel ou comparaison industrie)
#   3. Garder le format JSON structuré de la démonstration
prompt_modifie = f"""
# TODO : Écrivez votre prompt modifié ici
# Rappel : incluez les colonnes, l'aperçu, la contrainte, et le format JSON attendu
"""

# TODO : Envoyer le prompt à Ollama
# response = ollama.chat(...)

# TODO : Parser la réponse JSON (avec try/except comme dans la démo)
# Indice : même logique qu'en démonstration, avec re.search() en fallback

# TODO : Afficher les 5 recommandations de façon lisible
# Affichez au minimum : titre, type_chart, question_analytique, justification

---
## Exercice 3 — Génération de code Plotly

> **🎯 Objectif :** Demander au LLM de générer du code Python/Plotly, l'extraire et l'exécuter.

> **⏱️ Durée estimée : 40 minutes**

> **⚠️ Attention :** Le code généré par un LLM peut contenir des erreurs ! C'est normal et c'est précisément l'objet de la partie "À VOUS".

---
### 3-A — DÉMONSTRATION

In [ ]:
# NE PAS MODIFIER - Démonstration
# Fonction utilitaire pour extraire du code Python d'une réponse LLM

import re

def extract_python_code(text: str) -> str:
    """Extrait un bloc de code Python d'une réponse LLM.
    
    Stratégies dans l'ordre :
    1. Cherche un bloc ```python ... ```
    2. Cherche un bloc ``` ... ```
    3. Retourne le texte brut si aucun bloc trouvé
    """
    # Cherche un bloc ```python ... ```
    match = re.search(r'```python\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Sinon cherche ``` ... ```
    match = re.search(r'```\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Sinon retourne le texte brut
    return text.strip()

# Test de la fonction
texte_test = """
Voici le code que vous demandez :
```python
import plotly.express as px
fig = px.bar(x=[1,2,3], y=[4,5,6])
fig.show()
```
Ce code crée un graphique simple.
"""

code_extrait = extract_python_code(texte_test)
print("Code extrait avec succès :")
print(code_extrait)

In [ ]:
# NE PAS MODIFIER - Démonstration
# Génération et exécution d'un graphique Plotly via LLM

import ollama
import pandas as pd
import re

df = pd.read_csv('job_postings.csv')

# Préparation d'un résumé des données pour le contexte
top_industries = df['Company Industry'].value_counts().head(15).to_string()

# Prompt demandant la génération de code Plotly
prompt_code = f"""Tu es un expert Python et Plotly. Génère du code Python pour créer une visualisation.

Tâche : Créer un bar chart horizontal des 10 industries avec le plus d'offres d'emploi.

Contexte (données réelles du dataset) :
Top 15 industries par nombre d'offres :
{top_industries}

Contraintes :
- Utilise pandas et plotly.express
- Le DataFrame est déjà chargé dans la variable 'df' avec les colonnes : {', '.join(df.columns.tolist())}
- Crée un bar chart HORIZONTAL (orientation='h')
- Limite aux 10 premières industries
- Ajoute un titre en français
- Trie par ordre décroissant
- Termine avec fig.show()

Réponds UNIQUEMENT avec le bloc de code Python, encapsulé dans ```python ... ```, sans aucune explication."""

print("Génération du code Plotly via LLM...")
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt_code}]
)

contenu_brut = response['message']['content']
code_genere = extract_python_code(contenu_brut)

print("Code généré par le LLM :")
print("-" * 60)
print(code_genere)
print("-" * 60)

# --- Exécution du code avec exec() ---
# Note : exec() est puissant mais à utiliser avec précaution !
# On passe df dans le contexte d'exécution
print("\nExécution du code généré...")
try:
    exec(code_genere, {'df': df, 'pd': pd})
    print("Code exécuté avec succès !")
except Exception as e:
    print(f"Erreur lors de l'exécution : {e}")
    print("Le code généré par le LLM contient des erreurs — c'est un cas classique !")
    print("Consultez le code ci-dessus pour identifier et corriger manuellement le problème.")

---
### 3-B — À VOUS

> **📋 Consignes :**
> 1. Demandez au LLM de générer un graphique montrant la **distribution des années d'expérience (`Years of Experience`) par niveau de poste (`Job Position Level`)**
> 2. Extrayez le code avec `extract_python_code()`
> 3. Exécutez-le avec `exec()`
> 4. Si le code échoue, notez l'erreur, corrigez manuellement et relancez

> **💡 Conseil :** Un box plot (`px.box`) ou un violin plot (`px.violin`) seraient particulièrement adaptés ici. Vous pouvez le suggérer dans votre prompt.

> **💡 Conseil :** Vérifiez d'abord les valeurs uniques de `Years of Experience` avec `df['Years of Experience'].value_counts()` pour comprendre le format des données.

In [ ]:
# ✏️ À COMPLÉTER
# Exploration préalable des colonnes concernées
import pandas as pd

df = pd.read_csv('job_postings.csv')

# TODO : Affichez les valeurs uniques de 'Years of Experience' et 'Job Position Level'
# Indice : df['colonne'].value_counts() ou df['colonne'].unique()
# Cela vous aidera à rédiger un prompt précis

# print(df['Years of Experience'].value_counts())
# print(df['Job Position Level'].value_counts())

In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd
import plotly.express as px
import plotly.io as pio
import re, os

# Ouvre les graphiques dans le navigateur (contourne le problème fig.show() en Jupyter)
pio.renderers.default = 'browser'

df = pd.read_csv('job_postings.csv')

# Fonction utilitaire (déjà vue en 3-A)
def extract_python_code(text):
    match = re.search(r'```python\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r'```\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

# Fonction de réparation des erreurs LLM fréquentes sur Plotly
def repair_plotly_code(code: str) -> str:
    # Erreur fréquente : category_order= au lieu de category_orders= (dict)
    code = re.sub(r'category_order\s*=\s*\[', 'category_orders={', code)
    # Fermer le dict si le LLM a mis une liste
    code = re.sub(
        r'(category_orders=\{)(.*?)(\])',
        lambda m: m.group(1) + m.group(2) + '}',
        code, flags=re.DOTALL
    )
    # Remplacer fig.show() par sauvegarde HTML + ouverture navigateur
    code = code.replace('fig.show()', "fig.write_html('chart_ex3b.html')\nimport webbrowser; webbrowser.open('chart_ex3b.html')")
    return code

# TODO : Construire le prompt pour générer un box plot
# Il doit décrire :
#   - Le type de graphique souhaité (px.box)
#   - Les colonnes : 'Years of Experience' (axe Y) et 'Job Position Level' (axe X)
#   - Le fait que df est déjà disponible dans le contexte
#   - Le format de sortie : bloc ```python ... ```
niveaux = df['Job Position Level'].dropna().unique().tolist()
exp_range = f"{int(df['Years of Experience'].min())} à {int(df['Years of Experience'].max())}"

prompt_graphique = """
# TODO : Écrivez votre prompt ici en utilisant niveaux et exp_range
"""

# TODO : Appeler Ollama et extraire le code
# response = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': prompt_graphique}])
# code_genere = extract_python_code(response['message']['content'])

# TODO : Appliquer repair_plotly_code() avant exec()
# code_repare = repair_plotly_code(code_genere)
# print("Code réparé :")
# print(code_repare)

# TODO : Exécuter avec exec()
# try:
#     exec(code_repare, {'df': df, 'pd': pd, 'px': px})
#     print("\n✓ Graphique généré — vérifiez votre navigateur ou chart_ex3b.html")
# except Exception as e:
#     print(f"\n✗ Erreur résiduelle : {e}")
#     print("→ Corrigez manuellement et notez l'erreur en commentaire ci-dessous")
# Erreur rencontrée : ...
# Correction apportée : ...


---
## Exercice 4 — Observer les hallucinations

> **🎯 Objectif :** Comprendre empiriquement quand et comment les LLMs inventent des informations numériques.

> **⏱️ Durée estimée : 20 minutes**

---
### 4-A — DÉMONSTRATION

In [ ]:
# NE PAS MODIFIER - Démonstration
# Comparaison : LLM avec vs sans information numérique

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

# --- Scénario 1 : Sans information réelle ---
prompt_sans_contexte = """Un fichier job_postings.csv contient des offres d'emploi.
Combien d'offres d'emploi ce fichier contient-il selon toi ?
Donne un chiffre précis."""

print("SCÉNARIO 1 : LLM sans information réelle")
print("-" * 50)
r1 = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': prompt_sans_contexte}])
print(r1['message']['content'])

print()

# --- Scénario 2 : Avec l'information réelle dans le contexte ---
nb_lignes_reel = len(df)
prompt_avec_contexte = f"""Un fichier job_postings.csv contient des offres d'emploi.
Ce fichier a exactement {nb_lignes_reel} lignes (offres d'emploi).
Combien d'offres d'emploi ce fichier contient-il ?
Donne un chiffre précis."""

print("SCÉNARIO 2 : LLM avec l'information réelle")
print("-" * 50)
r2 = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': prompt_avec_contexte}])
print(r2['message']['content'])

print()
print("=" * 50)
print(f"VÉRITÉ : {nb_lignes_reel} lignes (source : pandas)")

In [ ]:
# NE PAS MODIFIER - Démonstration
# Le LLM invente une moyenne salariale

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

# On demande une moyenne sans donner les données
prompt_moyenne = """Dans un dataset d'offres d'emploi américain avec les colonnes
'Minimum Pay' et 'Maximum Pay' (en dollars annuels),
quel est selon toi le salaire minimum moyen et le salaire maximum moyen ?
Donne des chiffres précis."""

print("LLM : estimation du salaire moyen (SANS données réelles)")
print("-" * 50)
r = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': prompt_moyenne}])
print(r['message']['content'])

print()

# Calcul réel
sal_min_moyen = df['Minimum Pay'].dropna().mean()
sal_max_moyen = df['Maximum Pay'].dropna().mean()
print("VÉRITÉ TERRAIN (pandas) :")
print(f"  Salaire minimum moyen : {sal_min_moyen:,.2f} $")
print(f"  Salaire maximum moyen : {sal_max_moyen:,.2f} $")
print()
print(">>> Comparez les deux : l'écart illustre l'hallucination numérique.")

---
### 4-B — À VOUS

> **📋 Consignes :**
> 1. Concevez un prompt qui **force le LLM à dire "Je ne sais pas"** quand il n'a pas accès aux données numériques réelles
> 2. Testez ce prompt avec 3 questions numériques différentes sur le dataset
> 3. Notez dans les commentaires si la stratégie a fonctionné ou non

> **💡 Conseil :** Pour forcer l'honnêteté d'un LLM, essayez des formulations comme :
> - *"Si tu n'as pas accès aux données réelles, dis-le explicitement"*
> - *"Tu ne dois pas inventer de chiffres — si tu ne sais pas, réponds 'Donnée non disponible'"*
> - *"Ne génère que des statistiques que tu peux calculer à partir du contexte fourni"*

In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd

# Chargement du dataset (pour pouvoir vérifier les vraies valeurs après)
df = pd.read_csv('job_postings.csv')

# TODO : Définissez un système de prompt qui force le LLM à l'honnêteté numérique
# Indice : Un "system prompt" ou une instruction préalable dans le message utilisateur
instruction_honnêteté = """
# TODO : Rédigez une instruction qui oblige le LLM à dire "Je ne sais pas"
# quand il n'a pas accès aux données numériques réelles
"""

# TODO : Définissez 3 questions numériques à tester
questions_numeriques = [
    # TODO : Question 1 (ex: nombre d'offres dans un secteur précis)
    "Question 1 : ...",
    # TODO : Question 2 (ex: salaire médian)
    "Question 2 : ...",
    # TODO : Question 3 (ex: pourcentage d'offres en remote)
    "Question 3 : ...",
]

# TODO : Pour chaque question, envoyez le prompt avec l'instruction d'honnêteté
# et affichez la réponse
for i, question in enumerate(questions_numeriques, 1):
    print(f"\nQUESTION {i} : {question}")
    print("-" * 50)
    
    # TODO : Construire le prompt complet = instruction_honnêteté + question
    # prompt_complet = ...
    
    # TODO : Appeler Ollama et afficher la réponse
    # response = ollama.chat(...)
    # print(response['message']['content'])
    
    # TODO : Vérifiez la vraie valeur avec pandas et notez-la en commentaire
    # Vraie valeur (pandas) : ...
    # Le LLM a-t-il dit "Je ne sais pas" ? Oui / Non / Partiellement

---
## Exercice 5 — Few-shot prompting

> **🎯 Objectif :** Améliorer la qualité des recommandations en fournissant des exemples au LLM.

> **⏱️ Durée estimée : 25 minutes**

> **📖 Concept :** Le *few-shot prompting* consiste à inclure 2 ou 3 exemples de raisonnements réussis dans le prompt, avant de poser la vraie question. Le LLM s'aligne sur le style et la profondeur des exemples.

---
### 5-A — DÉMONSTRATION


In [ ]:
# NE PAS MODIFIER - Démonstration
# Few-shot : fournir 2 exemples d'analyses AVANT la vraie question

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')
colonnes = ', '.join(df.columns.tolist())

prompt_few_shot = f"""Tu es un expert en datavisualisation. Voici comment analyser un dataset :

--- EXEMPLE 1 ---
Dataset : ventes de produits (colonnes : date, produit, region, montant, quantite)
Analyse : La colonne 'date' permet une évolution temporelle. 'region' et 'produit' sont catégorielles.
Meilleure viz : Line chart de 'montant' par 'date', coloré par 'region' → montre les tendances de ventes par zone géographique.
Justification : Identifie les régions en croissance ou en déclin sur la durée.

--- EXEMPLE 2 ---
Dataset : résultats scolaires (colonnes : eleve_id, matiere, note, professeur, trimestre)
Analyse : 'matiere' et 'professeur' sont catégorielles, 'note' est numérique, 'trimestre' est temporel.
Meilleure viz : Box plot de 'note' par 'matiere' → montre la distribution des résultats par discipline.
Justification : Compare la dispersion des notes entre matières, révèle les matières difficiles.

--- VOTRE TOUR ---
Dataset : offres d'emploi (colonnes : {colonnes})
Analyse : """

print("Envoi du prompt few-shot...")
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt_few_shot}]
)
print("Réponse du LLM (guidée par les exemples) :")
print(response['message']['content'])


---
### 5-B — À VOUS

> **📋 Consignes :**
> 1. Rédigez **2 exemples** de recommandations de viz (inventez des datasets simples)
> 2. Demandez au LLM une recommandation pour le dataset `job_postings` avec ces 2 exemples
> 3. Comparez la réponse avec celle de l'Exercice 1-B (sans few-shot) : est-elle plus détaillée ? plus structurée ?

> **💡 Conseil :** Vos exemples doivent ressembler à ce que vous attendez en sortie. Si vous voulez une réponse avec un type de graphique, une colonne X/Y et une justification, vos exemples doivent contenir ces éléments.


In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

# TODO : Rédigez l'Exemple 1 (dataset inventé de votre choix)
exemple_1 = """
--- EXEMPLE 1 ---
Dataset : ...
Analyse : ...
Meilleure viz : ...
Justification : ...
"""

# TODO : Rédigez l'Exemple 2 (dataset inventé de votre choix)
exemple_2 = """
--- EXEMPLE 2 ---
Dataset : ...
Analyse : ...
Meilleure viz : ...
Justification : ...
"""

# TODO : Construisez le prompt few-shot complet avec vos 2 exemples
# puis posez la question sur job_postings
colonnes = ', '.join(df.columns.tolist())
prompt_few_shot = f"""
# TODO : combinez exemple_1, exemple_2, et la question sur job_postings
"""

# TODO : Envoyez le prompt et affichez la réponse
# response = ollama.chat(...)
# print(response['message']['content'])

# TODO : Comparez avec la réponse de l'exercice 1-B
# Notez 2 différences observées :
# Différence 1 : ...
# Différence 2 : ...


---
## Exercice 6 — Chain-of-thought : raisonner étape par étape

> **🎯 Objectif :** Comparer un prompt direct avec un prompt Chain-of-Thought (CoT) et mesurer l'impact sur la qualité du raisonnement.

> **⏱️ Durée estimée : 20 minutes**

> **📖 Concept :** Le CoT consiste à demander au LLM de *"penser à voix haute"* avant de donner sa réponse finale. Cette technique améliore la cohérence des réponses complexes car le LLM décompose le problème avant de conclure.

---
### 6-A — DÉMONSTRATION


In [ ]:
# NE PAS MODIFIER - Démonstration
# Comparaison prompt direct vs Chain-of-Thought

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')
apercu = df[['Job Title', 'Company Industry', 'Job Position Level', 'Years of Experience']].head(5).to_string(index=False)

question = f"""Dataset d'offres d'emploi :
{apercu}

Quelle visualisation recommandes-tu pour comprendre les exigences du marché de l'emploi ?"""

# --- Version 1 : prompt direct ---
print("=" * 60)
print("VERSION 1 — Prompt direct (sans CoT)")
print("=" * 60)
response_direct = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': question}]
)
print(response_direct['message']['content'])

# --- Version 2 : Chain-of-Thought ---
print("\n" + "=" * 60)
print("VERSION 2 — Chain-of-Thought (raisonnement guidé)")
print("=" * 60)
question_cot = question + """

Raisonne étape par étape :
Étape 1 : Identifie les types de colonnes (numérique, catégorielle, temporelle)
Étape 2 : Identifie les relations intéressantes entre colonnes
Étape 3 : Sélectionne le type de graphique le plus adapté
Étape 4 : Formule ta recommandation finale avec une justification"""

response_cot = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': question_cot}]
)
print(response_cot['message']['content'])


---
### 6-B — À VOUS

> **📋 Consignes :**
> 1. Posez une question complexe au LLM : *"Quelles 3 visualisations permettraient à un recruteur de comprendre les tendances du marché de l'emploi tech ?"*
> 2. Envoyez-la **sans** CoT, notez la réponse
> 3. Ajoutez des étapes de raisonnement, renvoyez, notez la réponse
> 4. Évaluez : le CoT a-t-il aidé ? Dans quel sens ?

> **💡 Conseil :** Des étapes utiles pour la dataviz : *"1/ Identifie le public cible, 2/ Liste les questions analytiques, 3/ Associe chaque question à un type de graphique"*


In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')
apercu = df[['Job Title', 'Company Industry', 'Job Position Level', 'Years of Experience',
             'Minimum Pay', 'Maximum Pay']].head(8).to_string(index=False)

question_base = f"""Dataset d'offres d'emploi tech :
{apercu}

Quelles 3 visualisations permettraient à un recruteur de comprendre les tendances du marché de l'emploi tech ?"""

# TODO : Envoyez la question SANS CoT et notez la réponse
# response_sans_cot = ollama.chat(...)
# print("Sans CoT :", response_sans_cot['message']['content'])

# TODO : Construisez la version CoT en ajoutant des étapes de raisonnement
# question_cot = question_base + """
# Raisonne étape par étape :
# Étape 1 : ...
# Étape 2 : ...
# Étape 3 : ...
# """
# response_cot = ollama.chat(...)
# print("Avec CoT :", response_cot['message']['content'])

# TODO : Évaluez les deux réponses
# Critère 1 — Pertinence des recommandations : Sans CoT [ /5] | Avec CoT [ /5]
# Critère 2 — Justifications détaillées     : Sans CoT [ /5] | Avec CoT [ /5]
# Critère 3 — Diversité des types de charts : Sans CoT [ /5] | Avec CoT [ /5]
# Conclusion : Le CoT a-t-il amélioré la réponse ? Pourquoi ?
# ...


---
## Exercice 7 — Effet de la température sur les réponses

> **🎯 Objectif :** Comprendre expérimentalement l'impact du paramètre `temperature` sur la créativité et la cohérence des réponses.

> **⏱️ Durée estimée : 20 minutes**

> **📖 Concept :** La température contrôle l'aléatoire de la génération. `temperature=0.1` → réponses déterministes et prudentes. `temperature=0.9` → réponses créatives mais moins prévisibles. `temperature=0.0` → toujours la même réponse.

---
### 7-A — DÉMONSTRATION


In [ ]:
# NE PAS MODIFIER - Démonstration
# Même prompt, 3 températures différentes → comparer les sorties

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

prompt = """Propose une visualisation originale et créative pour ce dataset d'offres d'emploi.
Colonnes disponibles : Job Title, Company Industry, Job Position Level, Years of Experience,
Minimum Pay, Maximum Pay, Number of Applicants, Job Skills.
Sois concis : type de chart, colonnes, et insight attendu en 3 lignes max."""

temperatures = [0.1, 0.5, 0.9]

for temp in temperatures:
    print(f"\n{'='*60}")
    print(f"TEMPÉRATURE : {temp}")
    print('='*60)
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temp}
    )
    print(response['message']['content'])


---
### 7-B — À VOUS

> **📋 Consignes :**
> 1. Envoyez le **même prompt** avec les températures `0.0`, `0.5` et `1.0`
> 2. Envoyez **deux fois** le prompt à `temperature=0.0` : les réponses sont-elles identiques ?
> 3. Envoyez **deux fois** le prompt à `temperature=1.0` : les réponses sont-elles différentes ?
> 4. Notez vos conclusions dans les commentaires

> **💡 Conseil :** Pour la génération de code Plotly (exercice 3), quelle température choisiriez-vous ? Justifiez.


In [ ]:
# ✏️ À COMPLÉTER
import ollama

# TODO : Rédigez un prompt court demandant une recommandation de viz
mon_prompt = """
# TODO : votre prompt ici (2-3 lignes suffisent)
"""

# TODO : Testez temperature=0.0 DEUX FOIS et comparez
print("=== TEMPÉRATURE 0.0 — Essai 1 ===")
# response_00_1 = ollama.chat(model='llama3.2', messages=[...], options={'temperature': 0.0})
# print(response_00_1['message']['content'])

print("\n=== TEMPÉRATURE 0.0 — Essai 2 ===")
# response_00_2 = ollama.chat(...)
# print(response_00_2['message']['content'])

# TODO : Testez temperature=0.5
print("\n=== TEMPÉRATURE 0.5 ===")
# response_05 = ollama.chat(...)
# print(response_05['message']['content'])

# TODO : Testez temperature=1.0 DEUX FOIS et comparez
print("\n=== TEMPÉRATURE 1.0 — Essai 1 ===")
# response_10_1 = ollama.chat(...)
# print(response_10_1['message']['content'])

print("\n=== TEMPÉRATURE 1.0 — Essai 2 ===")
# response_10_2 = ollama.chat(...)
# print(response_10_2['message']['content'])

# TODO : Complétez vos conclusions
# À temperature=0.0, les réponses sont-elles identiques ? ...
# À temperature=1.0, les réponses sont-elles différentes ? ...
# Pour générer du code Plotly fiable, quelle température choisir ? Pourquoi ? ...


---
## Exercice 8 — Limite de contexte : stratégies d'échantillonnage

> **🎯 Objectif :** Comprendre pourquoi on ne peut pas envoyer un grand dataset au LLM et maîtriser 3 stratégies pour le contourner.

> **⏱️ Durée estimée : 35 minutes**

> **📖 Concept :** Un LLM a une fenêtre de contexte limitée (ex: 8 000 à 128 000 tokens). Un CSV de 25 000 lignes dépasse largement cette limite. La solution : envoyer une **représentation intelligente** des données plutôt que les données brutes.

---
### 8-A — DÉMONSTRATION


In [ ]:
# NE PAS MODIFIER - Démonstration
# Stratégies pour contourner la limite de contexte

import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

question_analyse = "Quelles sont les tendances principales de ce dataset d'offres d'emploi ?"

# --- Stratégie 0 : envoi brut (à éviter) ---
print("STRATÉGIE 0 — Envoi brut (25 000 lignes)")
csv_brut = df.to_string()
print(f"  Taille du contexte : ~{len(csv_brut.split()):,} tokens estimés")
print(f"  Limite llama3.2   : ~8 000 tokens")
print(f"  → PROBLÈME : le contexte est tronqué ou l'appel échoue\n")

# --- Stratégie 1 : échantillon aléatoire ---
print("=" * 60)
print("STRATÉGIE 1 — Échantillon aléatoire (50 lignes)")
sample = df.sample(50, random_state=42).to_string(index=False, max_cols=8)
response1 = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': f"Dataset (50 lignes aléatoires) :\n{sample}\n\n{question_analyse}"}]
)
print(response1['message']['content'][:400])

# --- Stratégie 2 : résumé statistique ---
print("\n" + "=" * 60)
print("STRATÉGIE 2 — Résumé statistique (describe + value_counts)")
resume = f"""
Statistiques numériques :
{df.describe().to_string()}

Top 5 industries :
{df['Company Industry'].value_counts().head(5).to_string()}

Top 5 niveaux de poste :
{df['Job Position Level'].value_counts().head(5).to_string()}
"""
response2 = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': f"Résumé statistique du dataset :\n{resume}\n\n{question_analyse}"}]
)
print(response2['message']['content'][:400])


---
### 8-B — À VOUS

> **📋 Consignes :**
> 1. Implémentez une **3e stratégie** : les top-N valeurs les plus fréquentes pour chaque colonne catégorielle
> 2. Comparez les 3 stratégies sur la même question : *"Quelles compétences techniques sont les plus demandées ?"*
> 3. Notez laquelle donne la meilleure réponse et pourquoi

> **💡 Conseil :** Pour la question sur les compétences, la colonne `Job Skills` contient des listes séparées par des virgules. Vous pouvez les exploser avec `df['Job Skills'].str.split(', ').explode()` avant de faire un `value_counts()`.


In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd

df = pd.read_csv('job_postings.csv')

question = "Quelles compétences techniques sont les plus demandées dans ce dataset d'offres d'emploi ?"

# --- Stratégie 1 : échantillon aléatoire (50 lignes) ---
print("STRATÉGIE 1 — Échantillon aléatoire")
# TODO : prendre 50 lignes aléatoires, envoyer à Ollama, afficher réponse
# sample = df.sample(50, random_state=42)[['Job Title', 'Job Skills', 'Company Industry']].to_string(index=False)
# response1 = ollama.chat(...)
# print(response1['message']['content'][:300])

# --- Stratégie 2 : résumé statistique ---
print("\nSTRATÉGIE 2 — Résumé statistique")
# TODO : calculer les top 10 compétences avec str.split + explode + value_counts
# skills_series = df['Job Skills'].dropna().str.split(', ').explode()
# top_skills = skills_series.value_counts().head(20).to_string()
# response2 = ollama.chat(...)
# print(response2['message']['content'][:300])

# --- Stratégie 3 : top-N par colonne catégorielle (À IMPLÉMENTER) ---
print("\nSTRATÉGIE 3 — Top-N par colonne catégorielle")
# TODO : pour chaque colonne catégorielle, construire un résumé "top 5 valeurs"
colonnes_cat = ['Job Title', 'Company Industry', 'Job Position Level', 'Job Skills']
resume_topn = ""
for col in colonnes_cat:
    pass  # TODO : ajouter les top 5 valeurs de chaque colonne à resume_topn
# response3 = ollama.chat(...)
# print(response3['message']['content'][:300])

# TODO : Bilan comparatif
# Stratégie la plus pertinente pour cette question : ...
# Raison : ...


---
## Exercice 9 — LLM comme outil de transformation de données

> **🎯 Objectif :** Utiliser le LLM non plus pour recommander des visualisations, mais pour **enrichir et transformer** les données avant de les visualiser.

> **⏱️ Durée estimée : 40 minutes**

> **📖 Concept :** La colonne `Job Skills` contient des listes de compétences brutes (`"python, sql, agile, communication"`). Un LLM peut les catégoriser automatiquement en `technique` ou `soft skill`, créant une nouvelle colonne utile pour la visualisation.

---
### 9-A — DÉMONSTRATION


In [ ]:
# NE PAS MODIFIER - Démonstration
# Classifier des compétences en "technique" ou "soft skill" avec le LLM

import ollama
import pandas as pd
import json, re

df = pd.read_csv('job_postings.csv')

# On prend un petit échantillon pour la démo
sample_skills = df['Job Skills'].dropna().head(5).tolist()

def classify_skills(skills_str: str) -> dict:
    """Demande au LLM de classer les compétences d'une offre."""
    prompt = f"""Voici une liste de compétences d'une offre d'emploi : {skills_str}

Classe chaque compétence comme "technique" (langages, outils, frameworks, technologies)
ou "soft" (communication, leadership, travail en équipe, gestion, etc.).

Réponds UNIQUEMENT en JSON :
{{"technique": ["comp1", "comp2"], "soft": ["comp3"]}}"""

    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1}
    )
    contenu = response['message']['content']
    try:
        return json.loads(contenu)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', contenu, re.DOTALL)
        return json.loads(match.group()) if match else {"technique": [], "soft": []}

# Traitement des 5 premières lignes
print("Classification de 5 offres d'emploi...")
for i, skills in enumerate(sample_skills):
    result = classify_skills(skills)
    n_tech = len(result.get('technique', []))
    n_soft = len(result.get('soft', []))
    print(f"  Offre {i+1}: {n_tech} tech, {n_soft} soft — {skills[:60]}...")


---
### 9-B — À VOUS

> **📋 Consignes :**
> 1. Appliquez la fonction `classify_skills()` sur les **50 premières offres** du dataset
> 2. Ajoutez deux colonnes au dataframe : `nb_skills_tech` et `nb_skills_soft`
> 3. Créez un bar chart Plotly montrant le **ratio tech/soft par niveau de poste** (`Job Position Level`)
> 4. Interprétez : les postes seniors demandent-ils plus de soft skills que les juniors ?

> **⚠️ Attention :** Chaque appel au LLM prend 1-3 secondes. Sur 50 lignes, prévoyez 1-2 minutes de traitement. Affichez une progression avec `print(f"Traitement {i+1}/50...")`.


In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd
import plotly.express as px
import json, re

df = pd.read_csv('job_postings.csv')

# Reprise de la fonction de classification
def classify_skills(skills_str: str) -> dict:
    prompt = f"""Compétences : {skills_str}
Classe chaque compétence comme technique ou soft.
JSON uniquement : {{"technique": [...], "soft": [...]}}"""
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1}
    )
    contenu = response['message']['content']
    try:
        return json.loads(contenu)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', contenu, re.DOTALL)
        return json.loads(match.group()) if match else {"technique": [], "soft": []}

# TODO : Traiter les 50 premières lignes avec des compétences non nulles
df_sample = df.dropna(subset=['Job Skills', 'Job Position Level']).head(50).copy()

nb_tech_list = []
nb_soft_list = []

for i, skills in enumerate(df_sample['Job Skills']):
    print(f"Traitement {i+1}/50...", end='\r')
    result = classify_skills(skills)
    nb_tech_list.append(len(result.get('technique', [])))
    nb_soft_list.append(len(result.get('soft', [])))

# TODO : Ajouter les colonnes au dataframe
# df_sample['nb_skills_tech'] = nb_tech_list
# df_sample['nb_skills_soft'] = nb_soft_list
# df_sample['ratio_soft'] = df_sample['nb_skills_soft'] / (df_sample['nb_skills_tech'] + df_sample['nb_skills_soft'] + 1e-6)

# TODO : Calculer la moyenne du ratio soft par niveau de poste
# df_grouped = df_sample.groupby('Job Position Level')[['nb_skills_tech', 'nb_skills_soft', 'ratio_soft']].mean().reset_index()
# print(df_grouped)

# TODO : Créer un bar chart groupé tech vs soft par niveau de poste
# fig = px.bar(...)
# fig.show()

# TODO : Interprétation
# Les postes seniors (Director, Executive) ont-ils un ratio soft plus élevé ? ...
# Cela correspond-il à vos attentes ? ...


---
## Exercice 10 — Comparer deux modèles Ollama

> **🎯 Objectif :** Comprendre que le choix du modèle LLM est un paramètre de conception à part entière, avec des compromis qualité/vitesse/ressources.

> **⏱️ Durée estimée : 30 minutes**

> **📖 Concept :** Ollama permet d'exécuter plusieurs modèles localement. `llama3.2:3b` est léger et rapide (~2 Go). `mistral` est plus capable mais plus lent (~4 Go). Le meilleur modèle dépend de la tâche.

---
### 10-A — DÉMONSTRATION


In [ ]:
# NE PAS MODIFIER - Démonstration
# Comparer llama3.2 et mistral sur un prompt de recommandation

import ollama
import pandas as pd
import time

# Vérifier les modèles disponibles
print("Modèles Ollama installés :")
try:
    models = ollama.list()
    for m in models['models']:
        print(f"  - {m['name']}")
except Exception as e:
    print(f"  Erreur : {e}")

print("""
Si 'mistral' n'est pas installé, exécutez dans un terminal :
  ollama pull mistral
(~4 Go, quelques minutes de téléchargement)
""")

df = pd.read_csv('job_postings.csv')
apercu = df[['Job Title', 'Company Industry', 'Job Position Level', 'Years of Experience']].head(5).to_string(index=False)

prompt_test = f"""Dataset d'offres d'emploi :
{apercu}

Recommande une visualisation en JSON strict :
{{"type_chart": "...", "x": "...", "y": "...", "justification": "..."}}"""

modeles = ['llama3.2', 'mistral']

for modele in modeles:
    print(f"\n{'='*60}")
    print(f"MODÈLE : {modele}")
    print('='*60)
    debut = time.time()
    try:
        response = ollama.chat(
            model=modele,
            messages=[{'role': 'user', 'content': prompt_test}],
            options={'temperature': 0.1}
        )
        duree = time.time() - debut
        print(f"Temps de réponse : {duree:.1f}s")
        print(response['message']['content'])
    except Exception as e:
        print(f"Modèle non disponible : {e}")
        print("→ Installez-le avec : ollama pull " + modele)


---
### 10-B — À VOUS

> **📋 Consignes :**
> 1. Testez les deux modèles sur **3 prompts différents** : recommandation simple, génération de code Plotly, classification de compétences
> 2. Remplissez le tableau d'évaluation ci-dessous dans les commentaires
> 3. Recommandez un modèle pour chaque cas d'usage

> **📊 Tableau à compléter :**
> ```
> | Tâche                  | llama3.2 qualité | mistral qualité | llama3.2 temps | mistral temps |
> |------------------------|------------------|-----------------|----------------|---------------|
> | Recommandation viz     |      /5          |      /5         |      s         |      s        |
> | Génération code Plotly |      /5          |      /5         |      s         |      s        |
> | Classification skills  |      /5          |      /5         |      s         |      s        |
> ```


In [ ]:
# ✏️ À COMPLÉTER
import ollama
import pandas as pd
import time

df = pd.read_csv('job_postings.csv')
apercu = df[['Job Title', 'Company Industry', 'Job Position Level']].head(5).to_string(index=False)

modeles = ['llama3.2', 'mistral']

# --- Tâche 1 : Recommandation de visualisation ---
print("=== TÂCHE 1 : Recommandation de visualisation ===")
prompt_reco = f"Dataset : {apercu}\nRecommande une visualisation pertinente avec justification."

# TODO : Testez les deux modèles, mesurez le temps, notez la qualité
for modele in modeles:
    debut = time.time()
    # response = ollama.chat(model=modele, messages=[{'role': 'user', 'content': prompt_reco}])
    # duree = time.time() - debut
    # print(f"{modele} ({duree:.1f}s) : {response['message']['content'][:200]}")
    pass

# --- Tâche 2 : Génération de code Plotly ---
print("\n=== TÂCHE 2 : Génération de code Plotly ===")
# TODO : Demandez un bar chart des top 10 industries, testez les deux modèles
# prompt_code = "..."
# for modele in modeles:
#     ...

# --- Tâche 3 : Classification de compétences ---
print("\n=== TÂCHE 3 : Classification de compétences ===")
skills_test = df['Job Skills'].dropna().iloc[0]
# TODO : Demandez la classification tech/soft, testez les deux modèles
# prompt_class = f"Compétences : {skills_test}\nClassifie en technique ou soft skill. JSON uniquement."
# for modele in modeles:
#     ...

# TODO : Complétez le tableau comparatif en commentaire
# | Tâche                  | llama3.2 qualité | mistral qualité | llama3.2 temps | mistral temps |
# |------------------------|------------------|-----------------|----------------|---------------|
# | Recommandation viz     |      /5          |      /5         |      s         |      s        |
# | Génération code Plotly |      /5          |      /5         |      s         |      s        |
# | Classification skills  |      /5          |      /5         |      s         |      s        |

# TODO : Conclusion
# Modèle recommandé pour la génération de code : ...  Raison : ...
# Modèle recommandé pour la recommandation viz  : ...  Raison : ...
# Modèle recommandé pour la classification      : ...  Raison : ...


---
## Synthèse — Ce que vous avez appris

### Bilan : LLM pour la datavisualisation

| LLM est bon pour... | LLM ne remplace pas... |
|---|---|
| Suggérer des types de visualisations pertinentes | Le calcul de statistiques exactes |
| Générer du code Plotly boilerplate | La vérification des résultats numériques |
| Structurer une démarche analytique | L'exploration rigoureuse des données (EDA) |
| Interpréter qualitativement des tendances | La détection d'outliers et de valeurs manquantes |
| Reformuler des questions analytiques | pandas / numpy pour les agrégations réelles |
| Traduire des besoins métier en specs de viz | Le jugement visuel et le sens du design |
| Commenter et documenter du code | La validation des hypothèses statistiques |

---

### Points à retenir

1. **Ne jamais faire confiance aux chiffres d'un LLM sans vérification** : utilisez toujours pandas pour les statistiques réelles.

2. **Le prompting structuré (JSON) est essentiel** pour exploiter les réponses LLM de façon programmatique. Sans format imposé, les réponses sont imprévisibles.

3. **La fenêtre de contexte est une contrainte réelle** : n'envoyez pas les données brutes — envoyez un résumé pertinent (colonnes, types, statistiques, exemples).

4. **Le code généré est un point de départ, pas une solution finale** : il faut souvent corriger les noms de colonnes, les types, et la logique d'agrégation.

5. **La température impacte la déterminisme** : pour la génération de code ou de JSON, préférez une température basse (0.0–0.3).

6. **Ollama = confidentialité** : vos données ne quittent pas votre machine, ce qui est crucial pour des données sensibles en entreprise.

---

### Pour aller plus loin

- **LangChain** : framework pour orchestrer des chaînes de prompts complexes avec mémoire
- **LlamaIndex** : indexation de documents pour interroger des gros corpus via LLM
- **Ollama Function Calling** : permettre au LLM d'appeler des fonctions Python directement
- **Structured Output** (mode JSON natif d'Ollama) : `format='json'` dans `ollama.chat()` pour forcer une sortie JSON sans prompt trick
- **Modèles spécialisés code** : `codellama`, `deepseek-coder` pour de meilleures générations de code

---

*Séance 7 — LLM & Datavisualisation — UQAC M2 Visualisation de Données*